In [1]:
import os
import pandas as pd

DATASET_PATH = "dataset"

data = []

for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            filepath = os.path.join(folder, file)

            # Default labels
            label = {
                "filename": filepath,
                "acne_inflammatory": 0,
                "acne_noninflammatory": 0,
                "hyperpigmentation": 0,
                "wrinkles": 0
            }

            # Assign label based on folder
            if folder == "acne_inflammatory":
                label["acne_inflammatory"] = 1
            elif folder == "acne_noninflammatory":
                label["acne_noninflammatory"] = 1
            elif folder == "hyperpigmentation":
                label["hyperpigmentation"] = 1
            elif folder == "wrinkles":
                label["wrinkles"] = 1

            data.append(label)

df = pd.DataFrame(data)
df.to_csv("labels_final.csv", index=False)

print("CSV created successfully ✅")

CSV created successfully ✅


In [2]:
print(df.head())

                                            filename  acne_inflammatory  \
0  acne_inflammatory\07Acne0811011-Copy_jpg.rf.a9...                  1   
1  acne_inflammatory\07Acne081101_jpg.rf.78aec21a...                  1   
2  acne_inflammatory\07AcnePittedScars1-Copy_jpg....                  1   
3  acne_inflammatory\07AcnePittedScars_jpg.rf.515...                  1   
4  acne_inflammatory\07RosaceaFulFAce1-Copy_jpg.r...                  1   

   acne_noninflammatory  hyperpigmentation  wrinkles  
0                     0                  0         0  
1                     0                  0         0  
2                     0                  0         0  
3                     0                  0         0  
4                     0                  0         0  


In [3]:
print(df.sum())


filename                acne_inflammatory\07Acne0811011-Copy_jpg.rf.a9...
acne_inflammatory                                                    2060
acne_noninflammatory                                                 1970
hyperpigmentation                                                    2126
wrinkles                                                             1982
dtype: object


In [4]:
import os

sample_path = os.path.join("dataset", df.iloc[0]['filename'])
print(sample_path)
print(os.path.exists(sample_path))

dataset\acne_inflammatory\07Acne0811011-Copy_jpg.rf.a975d52a38f5b1880caaf3a5b23608ef.jpg
True


data generator

In [7]:

from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import os

class DataGenerator(Sequence):

    def __init__(self, df, batch_size=32):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, index):
        batch_df = self.df.iloc[index*self.batch_size:(index+1)*self.batch_size]

        images, labels = [], []

        for i in range(len(batch_df)):
            img_path = os.path.join("dataset", batch_df.iloc[i]['filename'])

            img = load_img(img_path, target_size=(224, 224))
            img = img_to_array(img) / 255.0   # ✅ IMPORTANT

            label = batch_df.iloc[i][[
                "acne_inflammatory",
                "acne_noninflammatory",
                "hyperpigmentation",
                "wrinkles"
            ]].values.astype(float)

            images.append(img)
            labels.append(label)

        return np.array(images), np.array(labels)

In [11]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze most layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(4, activation='sigmoid')   # 4 classes
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,619,332 (9.99 MB)

 Trainable params: 1,887,748 (7.20 MB)

 Non-trainable params: 731,584 (2.79 MB)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_gen = DataGenerator(train_df)
val_gen = DataGenerator(val_df)

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=[early_stop]
)

Epoch 1/25
203/203 ━━━━━━━━━━━━━━━━━━━━ 212s 962ms/step - AUC: 0.9552 - Precision: 0.8521 - Recall: 0.7662 - accuracy: 0.8310 - loss: 0.2284 - val_AUC: 0.9816 - val_Precision: 0.9246 - val_Recall: 0.8813 - val_accuracy: 0.9075 - val_loss: 0.1408
Epoch 2/25
203/203 ━━━━━━━━━━━━━━━━━━━━ 206s 1s/step - AUC: 0.9960 - Precision: 0.9647 - Recall: 0.9500 - accuracy: 0.9611 - loss: 0.0626 - val_AUC: 0.9831 - val_Precision: 0.9166 - val_Recall: 0.9069 - val_accuracy: 0.9087 - val_loss: 0.1458
Epoch 3/25
203/203 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - AUC: 0.9993 - Precision: 0.9857 - Recall: 0.9795 - accuracy: 0.9834 - loss: 0.0266 - val_AUC: 0.9823 - val_Precision: 0.9125 - val_Recall: 0.8931 - val_accuracy: 0.9056 - val_loss: 0.1512
Epoch 4/25
203/203 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - AUC: 0.9998 - Precision: 0.9957 - Recall: 0.9946 - accuracy: 0.9960 - loss: 0.0105 - val_AUC: 0.9789 - val_Precision: 0.9202 - val_Recall: 0.9081 - val_accuracy: 0.9094 - val_loss: 0.1747
Epoch 5/25
203/203 ━━━━━━

In [13]:
model.save("skin_model_final.h5")

In [15]:
from tensorflow.keras.models import load_model

model = load_model("skin_model_final.h5")
print("Model loaded ✅")

Model loaded ✅


In [16]:
classes = [
    "Acne_Inflammatory",
    "Acne_NonInflammatory",
    "Hyperpigmentation",
    "Wrinkles"
]

In [20]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

def predict_image(img_path, threshold=0.2):
    
    img = load_img(img_path, target_size=(224, 224))
    img = img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)[0]

    print("\nRaw Predictions:", preds)

    print("\nDetected Conditions:")
    found = False

    for i, p in enumerate(preds):
        if p > threshold:
            print(f"{classes[i]}: {p*100:.2f}%")
            found = True

    if not found:
        print("No major skin issue detected")

    return preds

In [21]:
predict_image("test_images/blackheades.jpeg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step

Raw Predictions: [8.6885852e-01 6.2207544e-01 9.1023931e-06 8.9775347e-07]

Detected Conditions:
Acne_Inflammatory: 86.89%
Acne_NonInflammatory: 62.21%


array([8.6885852e-01, 6.2207544e-01, 9.1023931e-06, 8.9775347e-07],
      dtype=float32)